In [97]:
import pandas as pd
import numpy as np

# Displaying the Data

- loading the data
- displaying number of entries
- displaying number of unique feature values

In [98]:
df = pd.read_csv("../data/raw.csv")

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1259 entries, 0 to 1258
Data columns (total 27 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   Timestamp                  1259 non-null   str  
 1   Age                        1259 non-null   int64
 2   Gender                     1259 non-null   str  
 3   Country                    1259 non-null   str  
 4   state                      744 non-null    str  
 5   self_employed              1241 non-null   str  
 6   family_history             1259 non-null   str  
 7   treatment                  1259 non-null   str  
 8   work_interfere             995 non-null    str  
 9   no_employees               1259 non-null   str  
 10  remote_work                1259 non-null   str  
 11  tech_company               1259 non-null   str  
 12  benefits                   1259 non-null   str  
 13  care_options               1259 non-null   str  
 14  wellness_program           1259 non

In [107]:
n = 60

data = {}
for col in df.columns:
    vc = df[col].value_counts(dropna=False).head(n)
    entries = [f"{val}: {cnt}" for val, cnt in vc.items()]
    entries += [''] * (n - len(entries))
    data[col] = entries

result = pd.DataFrame(data)
result

,Age,Country,self_employed,family_history,treatment,no_employees,remote_work,tech_company,benefits,care_options,...,leave,mental_health_consequence,phys_health_consequence,coworkers,supervisor,mental_health_interview,phys_health_interview,mental_vs_physical,obs_consequence,Gender_clean
0,29: 85,United States: 751,No: 1095,No: 767,Yes: 637,6-25: 290,No: 883,Yes: 1031,Yes: 477,No: 501,...,Don't know: 563,No: 490,No: 925,Some of them: 774,Yes: 516,No: 1008,Maybe: 557,Don't know: 576,No: 1075,Male: 995
1,32: 82,United Kingdom: 185,Yes: 146,Yes: 492,No: 622,26-100: 289,Yes: 376,No: 228,Don't know: 408,Yes: 444,...,Somewhat easy: 266,Maybe: 477,Maybe: 273,No: 260,No: 393,Maybe: 207,No: 500,Yes: 343,Yes: 184,Female: 247
2,31: 75,Canada: 72,Not Sure: 18,,,More than 1000: 282,,,No: 374,Not sure: 314,...,Very easy: 206,Yes: 292,Yes: 61,Yes: 225,Some of them: 350,Yes: 44,Yes: 202,No: 340,,Other: 17
3,26: 75,Germany: 45,,,,100-500: 176,,,,,...,Somewhat difficult: 126,,,,,,,,,
4,27: 71,Netherlands: 27,,,,1-5: 162,,,,,...,Very difficult: 98,,,,,,,,,
5,33: 70,Ireland: 27,,,,500-1000: 60,,,,,...,,,,,,,,,,
6,28: 68,Australia: 21,,,,,,,,,...,,,,,,,,,,
7,34: 65,France: 13,,,,,,,,,...,,,,,,,,,,
8,30: 63,India: 10,,,,,,,,,...,,,,,,,,,,
9,25: 61,New Zealand: 8,,,,,,,,,...,,,,,,,,,,


# Cleaning data

- deleting redundant features
- recategorizing open-form responses into defined categories
- median imputation for numerical features
- nan categorical feature inputting

In [100]:
df = df.drop(["comments", "state", "Timestamp", "work_interfere"], axis=1)

In [101]:
male_gender = ["Male", "male", "M", "m", "Make", "Male ",
               "Cis Male", "Man", "Male-ish", "maile",
               "something kinda male?", "Mal", "Male (CIS)",
               "Guy (-ish) ^_^", "male leaning androgynous",
               "msle", "Mail", "Malr", "Cis Man", "cis male",
                "ostensibly male, unsure what that really means"
                ]
female_gender = ["Female", "F", "f", "Woman",
                "female", "Female ", "Cis Female",
                "Femake", "woman", "cis-female/femme",
                "Female (cis)", "femail"
]
other_gender = [
                "Female (trans)", "Trans-female", "queer/she/they",
                "non-binary", "Nah", "All", "Enby", "fluid",
                "Genderqueer", "Androgyne", "Agender", "Trans woman",
                "Neuter", "queer", "A little about you", "p"
]

mapping = {**dict.fromkeys(male_gender, "Male"), **dict.fromkeys(female_gender, "Female"), **dict.fromkeys(other_gender, "Other")}
df['Gender_clean'] = df['Gender'].map(mapping)

df = df.drop(["Gender"], axis=1)

In [102]:
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
df.loc[(df["Age"] < 18) | (df["Age"] > 80), "Age"] = np.nan
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Age"] = df["Age"].astype(int)

In [106]:
df["self_employed"] = df["self_employed"].fillna("Not Sure")